# 03_llvip_thermal.ipynb — ImageBind-style Thermal Modality on LLVIP

Train a mini ImageBind-style model on LLVIP pairs: visible image and infrared thermal image. The visible image is the image anchor; the thermal branch is aligned into the same embedding space with symmetric InfoNCE.

This is a compact Kaggle notebook, not the full original ImageBind system.

## 1. Setup
Imports, seed setup, device detection, and output directories. All outputs stay under /kaggle/working/03_llvip_thermal_outputs.

In [ ]:
import os, glob, json, random, math, time
from pathlib import Path
from collections import defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from PIL import Image
try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

OUTPUT_DIR = Path('/kaggle/working/03_llvip_thermal_outputs')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
FIGURE_DIR = OUTPUT_DIR / 'figures'
RESULT_DIR = OUTPUT_DIR / 'results'
for d in [OUTPUT_DIR, CHECKPOINT_DIR, FIGURE_DIR, RESULT_DIR]: d.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

## 2. Config
Defaults target Kaggle T4/P100. Reduce epochs or batch size if running on CPU.

In [ ]:
BACKBONE = 'resnet18'
EMBED_DIM = 256
FREEZE_IMAGE_ENCODER = False
EPOCHS = 10
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.1
NUM_WORKERS = 2
IMAGE_SIZE = 224
RESIZE_SIZE = 256
IMAGE_EXTS = ['.jpg', '.jpeg', '.png', '.bmp']
COMMON_ROOTS = [Path('/kaggle/input/llvip-dataset'), Path('/kaggle/input/llvip_dataset'), Path('/kaggle/input/llvip')]
config = dict(BACKBONE=BACKBONE, EMBED_DIM=EMBED_DIM, FREEZE_IMAGE_ENCODER=FREEZE_IMAGE_ENCODER, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, TEMPERATURE=TEMPERATURE, NUM_WORKERS=NUM_WORKERS)
print(json.dumps(config, indent=2))

## 3. Dataset Discovery and Debug Tree
The root is detected from common LLVIP Kaggle paths or by scanning /kaggle/input. If missing, the notebook prints instructions instead of crashing.

In [ ]:
def print_tree(root, max_depth=2, max_items=100):
    root = Path(root)
    print('Debug tree:', root)
    if not root.exists():
        print('  [missing]'); return
    c = 0
    for p in sorted(root.rglob('*')):
        rel = p.relative_to(root)
        if len(rel.parts) > max_depth: continue
        print('  ' * len(rel.parts) + p.name + ('/' if p.is_dir() else ''))
        c += 1
        if c >= max_items:
            print('  ... truncated ...'); break

def find_llvip_root():
    candidates = []
    for p in COMMON_ROOTS:
        if p.exists(): candidates.append(p)
    ki = Path('/kaggle/input')
    if ki.exists(): candidates += [p for p in ki.iterdir() if p.is_dir()]
    candidates = list(dict.fromkeys(candidates))
    def score(root):
        try: names = [x.name.lower() for x in root.rglob('*') if x.is_dir()]
        except Exception: return 0
        return sum('visible' in n for n in names) + sum(('infrared' in n or 'thermal' in n or n == 'ir') for n in names) + sum('annotation' in n for n in names)
    scored = sorted([(score(c), c) for c in candidates], key=lambda x: x[0], reverse=True)
    if scored and scored[0][0] >= 2: return scored[0][1]
    print('LLVIP dataset not found. Add the LLVIP Dataset to this Kaggle notebook.')
    print('Expected examples: /kaggle/input/llvip-dataset, /kaggle/input/llvip_dataset, /kaggle/input/llvip')
    return None

DATA_ROOT = find_llvip_root()
print('DATA_ROOT:', DATA_ROOT)
if DATA_ROOT is not None: print_tree(DATA_ROOT, 2)

## 4. Pair Visible and Infrared Images
Images are paired by filename stem. If path names contain train, val, or test, that split is used. Otherwise a random 80/20 train/val split is created.

In [ ]:
def is_image_file(p): return Path(p).suffix.lower() in IMAGE_EXTS

def split_from_path(p):
    parts = [x.lower() for x in Path(p).parts]
    if any(x in parts for x in ['train','training']): return 'train'
    if any(x in parts for x in ['val','valid','validation']): return 'val'
    if any(x in parts for x in ['test','testing']): return 'test'
    return None

def modality_from_path(p):
    joined = '/'.join([x.lower() for x in Path(p).parts])
    if 'visible' in joined: return 'visible'
    if 'infrared' in joined or 'thermal' in joined: return 'thermal'
    return None

def discover_pairs(root):
    if root is None: return pd.DataFrame(columns=['stem','visible_path','thermal_path','split'])
    visible, thermal = {}, {}
    for p in Path(root).rglob('*'):
        if not (p.is_file() and is_image_file(p)): continue
        mod = modality_from_path(p)
        if mod is None: continue
        rec = {'path': str(p), 'split': split_from_path(p)}
        if mod == 'visible': visible[p.stem] = rec
        else: thermal[p.stem] = rec
    rows = []
    for stem in sorted(set(visible) & set(thermal)):
        rows.append({'stem': stem, 'visible_path': visible[stem]['path'], 'thermal_path': thermal[stem]['path'], 'split': visible[stem]['split'] or thermal[stem]['split']})
    df = pd.DataFrame(rows)
    if len(df) == 0: return df
    if df['split'].isna().all():
        rng = np.random.default_rng(SEED); idx = np.arange(len(df)); rng.shuffle(idx)
        split = np.array(['val'] * len(df), dtype=object); split[idx[:int(0.8*len(df))]] = 'train'; df['split'] = split
    else:
        df['split'] = df['split'].fillna('train')
        if 'val' not in set(df['split']):
            train_idx = df.index[df['split'] == 'train'].to_numpy(); np.random.default_rng(SEED).shuffle(train_idx)
            df.loc[train_idx[:max(1, int(0.2*len(train_idx)))], 'split'] = 'val'
        df.loc[df['split'] == 'test', 'split'] = 'val'
    return df.reset_index(drop=True)

pairs_df = discover_pairs(DATA_ROOT)
print('Pairs found:', len(pairs_df))
if len(pairs_df):
    print(pairs_df['split'].value_counts()); display(pairs_df.head())
else:
    print('No visible-infrared pairs found. Check folder names visible/infrared and matching filename stems.')
train_df = pairs_df[pairs_df['split'] == 'train'].reset_index(drop=True) if len(pairs_df) else pairs_df
val_df = pairs_df[pairs_df['split'] == 'val'].reset_index(drop=True) if len(pairs_df) else pairs_df
print('Train pairs:', len(train_df), 'Val pairs:', len(val_df))

## 5. Sample Pair Visualization
A few visible and infrared pairs are shown and saved for sanity checking.

In [ ]:
def show_sample_pairs(df, n=4):
    if len(df) == 0:
        print('No pairs to display.'); return
    sample = df.sample(min(n, len(df)), random_state=SEED)
    fig, axes = plt.subplots(len(sample), 2, figsize=(8, 3*len(sample)))
    if len(sample) == 1: axes = np.array([axes])
    for i, row in enumerate(sample.itertuples()):
        axes[i,0].imshow(Image.open(row.visible_path).convert('RGB')); axes[i,0].set_title('Visible anchor'); axes[i,0].axis('off')
        axes[i,1].imshow(Image.open(row.thermal_path).convert('RGB')); axes[i,1].set_title('Infrared thermal'); axes[i,1].axis('off')
    fig.tight_layout(); fig.savefig(FIGURE_DIR / 'llvip_thermal_sample_pairs.png', dpi=160, bbox_inches='tight'); plt.show()
show_sample_pairs(pairs_df, 4)

## 6. Dataset Class and Paired Transforms
Train augmentation uses the same crop and horizontal flip for visible and thermal images to preserve spatial alignment.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
visible_jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)

class PairedTrainTransform:
    def __init__(self, resize_size=256, crop_size=224): self.resize_size, self.crop_size = resize_size, crop_size
    def __call__(self, visible, thermal):
        visible = TF.resize(visible, [self.resize_size, self.resize_size], interpolation=InterpolationMode.BILINEAR)
        thermal = TF.resize(thermal, [self.resize_size, self.resize_size], interpolation=InterpolationMode.BILINEAR)
        i, j, h, w = transforms.RandomResizedCrop.get_params(visible, scale=(0.75, 1.0), ratio=(0.9, 1.1))
        visible = TF.resized_crop(visible, i, j, h, w, [self.crop_size, self.crop_size], interpolation=InterpolationMode.BILINEAR)
        thermal = TF.resized_crop(thermal, i, j, h, w, [self.crop_size, self.crop_size], interpolation=InterpolationMode.BILINEAR)
        if random.random() < 0.5: visible, thermal = TF.hflip(visible), TF.hflip(thermal)
        visible = visible_jitter(visible)
        return TF.normalize(TF.to_tensor(visible), IMAGENET_MEAN, IMAGENET_STD), TF.normalize(TF.to_tensor(thermal), IMAGENET_MEAN, IMAGENET_STD)

class PairedEvalTransform:
    def __init__(self, resize_size=256, crop_size=224): self.resize_size, self.crop_size = resize_size, crop_size
    def __call__(self, visible, thermal):
        visible = TF.center_crop(TF.resize(visible, [self.resize_size, self.resize_size]), [self.crop_size, self.crop_size])
        thermal = TF.center_crop(TF.resize(thermal, [self.resize_size, self.resize_size]), [self.crop_size, self.crop_size])
        return TF.normalize(TF.to_tensor(visible), IMAGENET_MEAN, IMAGENET_STD), TF.normalize(TF.to_tensor(thermal), IMAGENET_MEAN, IMAGENET_STD)

class LLVIPPairDataset(Dataset):
    def __init__(self, dataframe, transform=None): self.df, self.transform = dataframe.reset_index(drop=True).copy(), transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        visible = Image.open(row.visible_path).convert('RGB')
        thermal = Image.open(row.thermal_path).convert('RGB')
        if self.transform: visible, thermal = self.transform(visible, thermal)
        return {'visible_tensor': visible, 'thermal_tensor': thermal, 'visible_path': row.visible_path, 'thermal_path': row.thermal_path, 'stem': row.stem}

def collate_pairs(batch):
    return {'visible_tensor': torch.stack([b['visible_tensor'] for b in batch]), 'thermal_tensor': torch.stack([b['thermal_tensor'] for b in batch]), 'visible_path': [b['visible_path'] for b in batch], 'thermal_path': [b['thermal_path'] for b in batch], 'stem': [b['stem'] for b in batch]}

train_dataset = LLVIPPairDataset(train_df, PairedTrainTransform(RESIZE_SIZE, IMAGE_SIZE))
val_dataset = LLVIPPairDataset(val_df, PairedEvalTransform(RESIZE_SIZE, IMAGE_SIZE))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'), collate_fn=collate_pairs, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'), collate_fn=collate_pairs)
print('Batches train/val:', len(train_loader), len(val_loader))
if len(train_dataset):
    s = train_dataset[0]; print('Sample tensor shapes:', s['visible_tensor'].shape, s['thermal_tensor'].shape)

## 7. Model
A dual encoder maps visible and thermal images into a shared normalized embedding space. Pretrained torchvision weights are attempted, with automatic fallback to random initialization if Kaggle is offline.

In [ ]:
def build_resnet18(pretrained_try=True):
    if pretrained_try:
        try:
            m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            print('Loaded pretrained ResNet18 weights.'); return m
        except Exception as exc:
            print('Pretrained weights unavailable, using weights=None. Reason:', str(exc)[:250])
    return models.resnet18(weights=None)

class ResNetEncoder(nn.Module):
    def __init__(self, embed_dim=256, pretrained_try=True):
        super().__init__(); net = build_resnet18(pretrained_try); feat_dim = net.fc.in_features; net.fc = nn.Identity()
        self.backbone = net; self.proj = nn.Linear(feat_dim, embed_dim)
    def forward(self, x): return F.normalize(self.proj(self.backbone(x)), dim=-1)

class LLVIPThermalBindMini(nn.Module):
    def __init__(self, embed_dim=256, freeze_image_encoder=False):
        super().__init__(); self.visible_encoder = ResNetEncoder(embed_dim); self.thermal_encoder = ResNetEncoder(embed_dim)
        if freeze_image_encoder:
            for p in self.visible_encoder.backbone.parameters(): p.requires_grad = False
    def encode_visible(self, x): return self.visible_encoder(x)
    def encode_thermal(self, x): return self.thermal_encoder(x)
    def forward(self, visible, thermal): return self.encode_visible(visible), self.encode_thermal(thermal)

model = LLVIPThermalBindMini(EMBED_DIM, FREEZE_IMAGE_ENCODER).to(DEVICE)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 8. Symmetric InfoNCE Loss
The diagonal is the positive visible-thermal pair. The loss averages visible-to-thermal and thermal-to-visible cross entropy.

In [ ]:
def symmetric_infonce(image_emb, thermal_emb, temperature=0.1):
    logits = image_emb @ thermal_emb.T / temperature
    labels = torch.arange(image_emb.size(0), device=image_emb.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    i2t_acc = (logits.argmax(1) == labels).float().mean()
    t2i_acc = (logits.T.argmax(1) == labels).float().mean()
    return (loss_i2t + loss_t2i) / 2, i2t_acc, t2i_acc

if len(train_loader):
    b = next(iter(train_loader))
    with torch.no_grad():
        ve, te = model(b['visible_tensor'].to(DEVICE), b['thermal_tensor'].to(DEVICE))
        loss, ia, ta = symmetric_infonce(ve, te, TEMPERATURE)
    print('Sanity:', float(loss.cpu()), float(ia.cpu()), float(ta.cpu()))

## 9. Training
Uses AdamW, cosine LR, mixed precision on CUDA, and saves best plus last checkpoints. History is saved as JSON.

In [ ]:
def run_epoch(model, loader, optimizer=None, scaler=None, scheduler=None, train=True):
    model.train(train); totals = defaultdict(float)
    for batch in tqdm(loader, desc='train' if train else 'val', leave=False):
        visible = batch['visible_tensor'].to(DEVICE, non_blocking=True); thermal = batch['thermal_tensor'].to(DEVICE, non_blocking=True)
        if train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                ve, te = model(visible, thermal); loss, ia, ta = symmetric_infonce(ve, te, TEMPERATURE)
            if train:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update()
                if scheduler: scheduler.step()
        bs = visible.size(0); totals['n'] += bs; totals['loss'] += float(loss.detach().cpu()) * bs; totals['i2t'] += float(ia.cpu()) * bs; totals['t2i'] += float(ta.cpu()) * bs
    n = max(1, totals['n']); return {'loss': totals['loss']/n, 'i2t_acc': totals['i2t']/n, 't2i_acc': totals['t2i']/n}

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS * max(1, len(train_loader))))
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
history, best_val_loss, best_epoch = [], float('inf'), -1

if len(train_loader) == 0 or len(val_loader) == 0:
    print('Training skipped: train or val loader is empty. Add LLVIP dataset and rerun.')
else:
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time(); tr = run_epoch(model, train_loader, optimizer, scaler, scheduler, True); va = run_epoch(model, val_loader, train=False)
        row = {'epoch': epoch, 'train_loss': tr['loss'], 'train_i2t_acc': tr['i2t_acc'], 'train_t2i_acc': tr['t2i_acc'], 'val_loss': va['loss'], 'val_i2t_acc': va['i2t_acc'], 'val_t2i_acc': va['t2i_acc'], 'lr': optimizer.param_groups[0]['lr'], 'seconds': round(time.time()-t0, 2)}
        history.append(row); print(json.dumps(row, indent=2))
        if va['loss'] < best_val_loss:
            best_val_loss, best_epoch = va['loss'], epoch
            torch.save({'model_state_dict': model.state_dict(), 'config': config, 'epoch': epoch, 'history': history}, CHECKPOINT_DIR / 'best_llvip_thermal.pt')
            print('Saved best checkpoint')
    torch.save({'model_state_dict': model.state_dict(), 'config': config, 'epoch': EPOCHS, 'history': history}, CHECKPOINT_DIR / 'last_llvip_thermal.pt')

with open(RESULT_DIR / 'llvip_thermal_train_history.json', 'w') as f: json.dump(history, f, indent=2)
print('Best epoch:', best_epoch, 'best val loss:', best_val_loss)

## 10. Retrieval Evaluation
Extract validation embeddings and compute Recall@1, Recall@5, and Recall@10 in both directions.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader):
    model.eval(); ves, tes, stems, vp, tp = [], [], [], [], []
    for batch in tqdm(loader, desc='extract embeddings'):
        ve, te = model(batch['visible_tensor'].to(DEVICE), batch['thermal_tensor'].to(DEVICE))
        ves.append(ve.cpu()); tes.append(te.cpu()); stems += batch['stem']; vp += batch['visible_path']; tp += batch['thermal_path']
    if not ves: return None
    return {'visible_emb': torch.cat(ves), 'thermal_emb': torch.cat(tes), 'stems': stems, 'visible_paths': vp, 'thermal_paths': tp}

def recall_at_k(sim, ks=(1,5,10)):
    n = sim.size(0); ranked = sim.topk(min(max(ks), n), dim=1).indices; target = torch.arange(n).view(-1,1); out = {}
    for k in ks:
        kk = min(k, n); out[f'R@{k}'] = float((ranked[:,:kk] == target).any(1).float().mean().item())
    return out

embeds = extract_embeddings(model, val_loader) if len(val_loader) else None
if embeds is None:
    retrieval_results = {}; print('No validation embeddings available.')
else:
    sim = embeds['visible_emb'] @ embeds['thermal_emb'].T; i2t = recall_at_k(sim); t2i = recall_at_k(sim.T)
    retrieval_results = {'image_to_thermal_R@1': i2t['R@1'], 'image_to_thermal_R@5': i2t['R@5'], 'image_to_thermal_R@10': i2t['R@10'], 'thermal_to_image_R@1': t2i['R@1'], 'thermal_to_image_R@5': t2i['R@5'], 'thermal_to_image_R@10': t2i['R@10'], 'num_val_pairs': len(embeds['stems'])}
    display(pd.DataFrame([retrieval_results]).T.rename(columns={0:'value'}))
with open(RESULT_DIR / 'llvip_thermal_retrieval_recall.json', 'w') as f: json.dump(retrieval_results, f, indent=2)

## 11. Optional Binary Person/Background Evaluation
If XML annotations are available, person and background thermal crops are created. A frozen thermal encoder plus linear classifier is trained briefly. Missing annotations are skipped gracefully.

In [ ]:
def find_annotation_dir(root):
    if root is None: return None
    dirs = [p for p in Path(root).rglob('*') if p.is_dir() and 'annotation' in p.name.lower()]
    return dirs[0] if dirs else None

def parse_xml_boxes(xml_path):
    boxes = []
    try:
        root = ET.parse(xml_path).getroot()
        for obj in root.findall('.//object'):
            b = obj.find('bndbox')
            if b is None: continue
            box = tuple(int(float(b.findtext(k, '0'))) for k in ['xmin','ymin','xmax','ymax'])
            if box[2] > box[0] and box[3] > box[1]: boxes.append(box)
    except Exception: return []
    return boxes

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b; ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); aa=max(1,(ax2-ax1)*(ay2-ay1)); ba=max(1,(bx2-bx1)*(by2-by1)); return inter/max(1,aa+ba-inter)

class BinaryCropDataset(Dataset):
    def __init__(self, records, transform): self.records, self.transform = records, transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]; img = Image.open(r['path']).convert('RGB').crop(r['box']); return self.transform(img), torch.tensor(r['label']).long()

binary_results = None
ann_dir = find_annotation_dir(DATA_ROOT)
if ann_dir is None or len(pairs_df) == 0:
    print('Binary eval skipped: annotations missing or pairs unavailable.')
else:
    records, rng = [], random.Random(SEED)
    for row in pairs_df.sample(min(len(pairs_df), 600), random_state=SEED).itertuples():
        xmls = list(Path(ann_dir).rglob(row.stem + '.xml'))
        if not xmls: continue
        boxes = parse_xml_boxes(xmls[0])
        if not boxes: continue
        W,H = Image.open(row.thermal_path).size
        for box in boxes[:3]:
            x1,y1,x2,y2 = max(0,box[0]),max(0,box[1]),min(W,box[2]),min(H,box[3])
            if x2-x1 < 12 or y2-y1 < 12: continue
            records.append({'path': row.thermal_path, 'box': (x1,y1,x2,y2), 'label': 1})
            bw,bh=x2-x1,y2-y1
            for _ in range(20):
                nx1=rng.randint(0,max(0,W-bw)); ny1=rng.randint(0,max(0,H-bh)); nb=(nx1,ny1,nx1+bw,ny1+bh)
                if all(iou(nb,b)<0.1 for b in boxes): records.append({'path': row.thermal_path, 'box': nb, 'label': 0}); break
    print('Binary crop records:', len(records))
    if len(records) < 20:
        print('Binary eval skipped: not enough parsed crops.')
    else:
        rng.shuffle(records); ntr=int(0.8*len(records)); tfm=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        tr=DataLoader(BinaryCropDataset(records[:ntr],tfm), batch_size=64, shuffle=True, num_workers=NUM_WORKERS); va=DataLoader(BinaryCropDataset(records[ntr:],tfm), batch_size=64)
        for p in model.thermal_encoder.parameters(): p.requires_grad=False
        clf=nn.Linear(EMBED_DIM,2).to(DEVICE); opt=torch.optim.AdamW(clf.parameters(), lr=1e-3)
        for ep in range(1,4):
            clf.train()
            for x,y in tqdm(tr, desc=f'binary train {ep}', leave=False):
                x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
                with torch.no_grad(): emb=model.encode_thermal(x)
                loss=F.cross_entropy(clf(emb), y); loss.backward(); opt.step()
        preds, labels = [], []; clf.eval()
        with torch.no_grad():
            for x,y in va:
                preds += clf(model.encode_thermal(x.to(DEVICE))).argmax(1).cpu().tolist(); labels += y.tolist()
        acc=float(np.mean(np.array(preds)==np.array(labels))) if labels else 0.0; cm=np.zeros((2,2), dtype=int)
        for y,p in zip(labels,preds): cm[y,p]+=1
        binary_results={'accuracy':acc,'top1':acc,'num_val_crops':len(labels),'confusion_matrix':cm.tolist()}
        with open(RESULT_DIR/'llvip_thermal_binary_eval_results.json','w') as f: json.dump(binary_results,f,indent=2)
        fig,ax=plt.subplots(figsize=(4,4))
        if HAS_SEABORN: sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['bg','person'], yticklabels=['bg','person'], ax=ax)
        else: ax.imshow(cm, cmap='Blues')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Thermal binary confusion matrix'); fig.tight_layout(); fig.savefig(FIGURE_DIR/'llvip_thermal_confusion_matrix.png', dpi=160); plt.show()
        print(binary_results)

## 12. Visualization
Plots learning curves and qualitative retrieval examples in both directions.

In [ ]:
def plot_learning_curve(history):
    if not history: print('No history to plot.'); return
    df=pd.DataFrame(history); fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].plot(df.epoch,df.train_loss,label='train'); axes[0].plot(df.epoch,df.val_loss,label='val'); axes[0].set_title('Loss')
    axes[1].plot(df.epoch,df.train_i2t_acc,label='train'); axes[1].plot(df.epoch,df.val_i2t_acc,label='val'); axes[1].set_title('Visible to thermal top1')
    axes[2].plot(df.epoch,df.train_t2i_acc,label='train'); axes[2].plot(df.epoch,df.val_t2i_acc,label='val'); axes[2].set_title('Thermal to visible top1')
    for ax in axes: ax.grid(alpha=.3); ax.legend(); ax.set_xlabel('Epoch')
    fig.tight_layout(); fig.savefig(FIGURE_DIR/'llvip_thermal_learning_curve.png', dpi=160, bbox_inches='tight'); plt.show()
plot_learning_curve(history)

def plot_retrieval_examples(embeds, n_queries=5, top_k=5):
    if embeds is None: print('No embeddings to visualize.'); return
    sim=embeds['visible_emb']@embeds['thermal_emb'].T; n=min(n_queries, sim.size(0)); fig,axes=plt.subplots(n*2, top_k+1, figsize=(2.4*(top_k+1),3.8*n))
    if n==1: axes=np.expand_dims(axes,0)
    for qi in range(n):
        row=qi*2; axes[row,0].imshow(Image.open(embeds['visible_paths'][qi]).convert('RGB')); axes[row,0].set_title('Visible query'); axes[row,0].axis('off')
        for j,idx in enumerate(sim[qi].topk(min(top_k,sim.size(1))).indices.tolist(),1): axes[row,j].imshow(Image.open(embeds['thermal_paths'][idx]).convert('RGB')); axes[row,j].set_title(f'Thermal #{j}'); axes[row,j].axis('off')
        row=qi*2+1; axes[row,0].imshow(Image.open(embeds['thermal_paths'][qi]).convert('RGB')); axes[row,0].set_title('Thermal query'); axes[row,0].axis('off')
        for j,idx in enumerate(sim[:,qi].topk(min(top_k,sim.size(0))).indices.tolist(),1): axes[row,j].imshow(Image.open(embeds['visible_paths'][idx]).convert('RGB')); axes[row,j].set_title(f'Visible #{j}'); axes[row,j].axis('off')
    fig.tight_layout(); fig.savefig(FIGURE_DIR/'llvip_thermal_retrieval_examples.png', dpi=150, bbox_inches='tight'); plt.show()
plot_retrieval_examples(embeds, 5, 5)

## 13. Temperature Ablation
A quick temperature sweep recomputes retrieval metrics from validation embeddings and saves JSON plus a plot.

In [ ]:
temps=[0.05,0.1,0.2,0.5]; ablation=[]
if embeds is None: print('Temperature ablation skipped: no embeddings.')
else:
    base=embeds['visible_emb']@embeds['thermal_emb'].T
    for temp in temps:
        i2t=recall_at_k(base/temp); t2i=recall_at_k((base/temp).T)
        ablation.append({'temperature':temp,'image_to_thermal_R@1':i2t['R@1'],'image_to_thermal_R@5':i2t['R@5'],'image_to_thermal_R@10':i2t['R@10'],'thermal_to_image_R@1':t2i['R@1'],'thermal_to_image_R@5':t2i['R@5'],'thermal_to_image_R@10':t2i['R@10']})
    ab_df=pd.DataFrame(ablation); display(ab_df); fig,ax=plt.subplots(figsize=(7,4)); ax.plot(ab_df.temperature, ab_df['image_to_thermal_R@1'], marker='o', label='I2T R@1'); ax.plot(ab_df.temperature,ab_df['thermal_to_image_R@1'], marker='o', label='T2I R@1'); ax.set_xlabel('Temperature'); ax.set_ylabel('Recall@1'); ax.grid(alpha=.3); ax.legend(); fig.tight_layout(); fig.savefig(FIGURE_DIR/'llvip_thermal_temperature_ablation.png', dpi=160); plt.show()
with open(RESULT_DIR/'llvip_thermal_temperature_ablation.json','w') as f: json.dump(ablation,f,indent=2)

## 14. Report
A text report summarizes dataset, model, training, retrieval metrics, optional binary metrics, notes, and future work.

In [ ]:
def fmt(d): return 'No metrics available.' if not d else '\n'.join([f'{k}: {v}' for k,v in d.items()])
report = f'''
03_llvip_thermal.ipynb report

Dataset summary
- Root: {DATA_ROOT}
- Total visible-thermal pairs: {len(pairs_df)}
- Train pairs: {len(train_df)}
- Validation pairs: {len(val_df)}

Model summary
- Simplified ImageBind-style visible-thermal dual encoder
- Visible image is the image anchor
- Backbone: {BACKBONE}
- Embedding dim: {EMBED_DIM}
- Freeze image encoder: {FREEZE_IMAGE_ENCODER}

Training config
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}
- LR: {LR}
- Weight decay: {WEIGHT_DECAY}
- Temperature: {TEMPERATURE}
- Device: {DEVICE}

Best epoch
- Best epoch: {best_epoch}
- Best val loss: {best_val_loss}

Final retrieval metrics
{fmt(retrieval_results)}

Optional binary classification metrics
{fmt(binary_results) if binary_results is not None else 'Skipped or unavailable.'}

Notes
- Visible image is used as the anchor.
- Thermal is aligned into the same embedding space by symmetric InfoNCE.
- Results may be limited by dataset size, augmentations, batch size, and Kaggle GPU limits.

Future work
- Train longer.
- Use ViT instead of ResNet.
- Use OpenCLIP/ImageBind pretrained image encoder.
- Improve annotation parsing for person/background classification.
'''.strip()
report_path=OUTPUT_DIR/'03_llvip_thermal_report.txt'
with open(report_path,'w') as f: f.write(report)
print(report); print('\nSaved report:', report_path)

## 15. Output Files
Final list of created artifacts.

In [ ]:
print('Files under', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file(): print(p)